# 01 · Caso de negocio — RiskVision

**Grupo 6 (Analistas de Datos):** Juan David Ardila Arboleda, Lina Isabel Vasquez Villegas, Andres Felipe Orozco Roa, Cristina Garces y Estefania Cervera.
**Dataset:** *Default of Credit Card Clients* (UCI Machine Learning Repository, licencia CC BY 4.0), 30.000 clientes de un banco de Taiwán.

## Para qué sirve este notebook
El proyecto necesita tres cosas escritas **con números**: **una pregunta de negocio, una métrica que la refleje y una hipótesis del costo de equivocarse**. Esta revisión conserva resultados exploratorios previos; no representa un planteamiento anterior a toda consulta de los datos.
Este notebook las define y su resumen va a `reports/00_caso.md`.

## Cómo leerlo
Cada sección responde una pregunta. Las celdas de código hacen cuentas muy cortas y debajo de cada una hay una **lectura** en palabras simples.

## Palabras clave
| Término | Significado simple |
|---|---|
| **Falso negativo** | El modelo dice «paga» y el cliente **no paga**. Se omite una revisión preventiva; se supone el error más caro |
| **Falso positivo** | El modelo dice «incumple» y el cliente **sí pagaba**. Genera una revisión innecesaria y posibles molestias |
| **Matriz de costos** | Tabla que dice cuánto cuesta cada tipo de error, en unidades relativas |
| **Línea base** | Resultado de una estrategia sin modelo. El modelo solo vale la pena si la supera |
| **Umbral** | Nivel de riesgo desde el cual se decide marcar a un cliente |


## 1. La pregunta de negocio

**Pregunta de negocio:** ¿Qué clientes de tarjeta de crédito tienen alta probabilidad de no pagar el próximo mes?

**¿Quién decide y con qué información?** El área de riesgo de crédito del banco. Cuenta con el límite de crédito, datos demográficos, el estado de pago de los últimos 6 meses y los montos facturados y pagados. Con la probabilidad de impago de cada cliente decide si ajusta el límite, hace seguimiento preventivo u ofrece reestructurar la deuda.

**¿Qué pasa hoy, sin el modelo?** Sin modelo no hay un orden de prioridad por riesgo: el equipo revisa a los clientes con reglas simples o sin criterio uniforme, y no sabe a quién contactar primero. *(Descripción general del caso: los datos no dicen cómo trabaja hoy el banco; el grupo debe confirmarlo.)*


## 2. Equivocarse no cuesta lo mismo

**Error más caro bajo la hipótesis del grupo:** no detectar a un cliente que sí va a incumplir (falso negativo). Se pierde la oportunidad de revisar su caso a tiempo; el dataset no permite cuantificar la pérdida ni asumir que equivale a todo el dinero prestado.

**Error menos caro bajo esa hipótesis:** marcar como riesgoso a un cliente que sí iba a pagar (falso positivo). Para esta evaluación se considera una revisión preventiva innecesaria. Ajustar límites o reestructurar son decisiones distintas, cuyos costos y efectos no están medidos aquí.

**Proporción y de dónde sale:** **5 a 1 es una hipótesis económica del grupo, no un costo observado del banco ni un dato del dataset**. Se usa como penalización relativa provisional y debe contrastarse con el instructor o con información económica del banco. Acertar tiene penalización 0 en esta simplificación: detectar un impago no demuestra que se evite la pérdida ni que intervenir sea gratis. No se han medido recuperaciones, eficacia de las acciones ni costos de atención de todos los clientes marcados.


**Qué hace la celda:** escribe la matriz de penalizaciones relativas. Cada fila es lo que realmente pasó y cada columna lo que predijo el modelo. Acertar recibe 0; una falsa alarma, 1; no detectar a quien incumple, 5. Cambiar el `5` y ejecutar las celdas de código recalcula las líneas base; los textos y la tabla exploratoria no se actualizan automáticamente.

In [1]:
import numpy as np
import pandas as pd

CLASSES = ["paga", "incumple"]

# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Diagonal = 0.
# SUPUESTO: no detectar un "incumple" cuesta 5 veces más que una falsa alarma.
COST_MATRIX = [[0, 1], [5, 0]]

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
display(cost)

,predice paga,predice incumple
real paga,0,1
real incumple,5,0


## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Calcula el costo de **dos estrategias sin modelo**:
- **Siempre la clase más frecuente** (o, en regresión, siempre el promedio).
- **Azar según las proporciones.**

Las proporciones reales vienen del conjunto de datos completo (30.000 clientes): 77,88 % paga y 22,12 % incumple.


**Qué hace la celda:** toma la proporción real de clientes que pagan (77,88 %) y que incumplen (22,12 %), y calcula cuánto costaría **en promedio por cliente** cada estrategia sin modelo usando la matriz de arriba. Un número menor es mejor.

In [2]:
# Proporciones en el mismo orden que CLASSES: 23.364 pagan y 6.636 incumplen, de 30.000 clientes.
priors = np.array([23364, 6636]) / 30000

cm = np.array(COST_MATRIX, dtype=float)
most_frequent = int(priors.argmax())
always_most_frequent = priors @ cm[:, most_frequent]
random_by_priors = priors @ cm @ priors
always_flag = priors @ cm[:, 1]
print(f"Siempre '{CLASSES[most_frequent]}':  costo esperado = {always_most_frequent:.2f}")
print(f"Azar según proporciones:   costo esperado = {random_by_priors:.2f}")
print(f"(Referencia) Marcar a todos como 'incumple': costo esperado = {always_flag:.2f}")

Siempre 'paga':  costo esperado = 1.11
Azar según proporciones:   costo esperado = 1.03
(Referencia) Marcar a todos como 'incumple': costo esperado = 0.78


**Lo que dice la línea base (una frase):** Un modelo que siempre dice «paga» acierta 77,9 % de las veces pero no detecta a ningún cliente que incumple, y con la penalización 5 a 1 su costo relativo esperado (1,11) es peor que el del azar (1,03). Marcar a **todos** como «incumple» cuesta aún menos (0,78), porque no tiene falsos negativos: es la mejor de estas estrategias triviales bajo la matriz asumida. Una evaluación útil debe comparar todas las alternativas sobre las mismas observaciones y considerar la capacidad de revisión; estas cifras no representan dinero ni ahorro demostrado.

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** No. Con 77,9 % de clientes que pagan, el modelo trivial ya tiene 77,9 % de exactitud sin detectar un solo incumplimiento, y la exactitud trata igual los dos errores aunque no cuesten lo mismo.


**Una línea base adicional (sección 6 del notebook 02):** una regla que marca a quien ya tiene 2 o más meses de atraso tiene penalización media 0,775 sobre el dataset completo, casi lo mismo que marcar a todos. Es una referencia exploratoria: la comparación posterior con modelos debe hacerse sobre las mismas observaciones de validación y con la misma matriz.

## 4. Cómo se medirá el éxito

- **Métrica principal: ROC AUC.** Mide qué tan bien el modelo **ordena** a los clientes de menor a mayor riesgo, sin fijar todavía un umbral. Interpretación: si se toma al azar un cliente que incumple y otro que no, el AUC es la probabilidad de que el modelo le asigne más riesgo al que incumple. 0,5 es azar y 1,0 es perfecto. Se eligió porque las clases están desbalanceadas y los datos no traen costos para justificar un umbral.
- **Métricas de control** (miran el error que más cuesta):

| Métrica | Qué responde | Por qué importa aquí |
|---|---|---|
| **Recall** de «incumple» | De los que realmente incumplen, ¿a cuántos detectamos? | Es el error caro (falso negativo) |
| **Precision** de «incumple» | De los que marcamos, ¿cuántos realmente incumplen? | Mide cuántas falsas alarmas genera el equipo |
| **F1** | Equilibrio entre recall y precision | Resumen cuando ambos importan |
| **AUC-PR** | Calidad del ranking enfocada en la clase rara | Con 22 % de incumplimiento es más exigente que el AUC. La referencia al azar es 0,22 |
| **Costo relativo por cliente** | Con la matriz asumida, ¿cuál es la penalización media por equivocarse? | Permite comparar decisiones; no mide dinero ni ahorro real |
| ~~Exactitud~~ | No usarla | «Siempre paga» ya acierta 77,9 % sin detectar a nadie |

- **Criterio provisional de utilidad:** superar el ranking al azar y reducir la penalización relativa frente a las líneas base sobre las mismas observaciones de validación, informando variabilidad y porcentaje de clientes marcados. Esto no basta para demostrar impacto económico; la meta operativa debe acordarse con el instructor.

**Umbral teórico bajo 5 a 1:** si `p` es la probabilidad calibrada de incumplimiento, decidir «paga» tiene costo esperado `5p` y decidir «incumple», `1 - p`. Se igualan en `p = 1/6 ≈ 0,167`; por encima se prefiere marcar y por debajo no marcar. En la igualdad ambas decisiones tienen el mismo costo. Esta regla depende de probabilidades calibradas para la población de uso y de la matriz `[[0, 1], [5, 0]]`, sin otras restricciones o costos de intervención. No demuestra que las probabilidades del modelo actual estén calibradas ni fija un umbral operativo definitivo.

**Umbral seleccionado posteriormente:** se decidirá utilizando exclusivamente validación dentro del entrenamiento original, con costos y criterios definidos de antemano. Se comprobará la calibración del modelo concreto y se comparará con la referencia teórica. No se usará la prueba original para elegirlo.

**Resultados exploratorios conservados:** la tabla siguiente ya figuraba en este notebook como referencia atribuida al Gradient Boosting optimizado. El procedimiento que produjo sus umbrales, métricas y costos no está documentado de forma reproducible en los archivos revisados. Se conservan las cifras, pendientes de verificación; no se afirma que los umbrales se hayan seleccionado correctamente mediante validación cruzada.

| Costo de no detectar vs falsa alarma | Umbral reportado (exploratorio) | Recall | Precision | Clientes marcados | Costo relativo modelo | Siempre «paga» | Marcar a todos |
|---|---|---|---|---|---|---|---|
| 2 a 1 | 0,35 | 48,5 % | 57,9 % | 18,6 % | **0,31** | 0,44 | 0,78 |
| **5 a 1** (supuesto) | 0,17 | 70,2 % | 39,3 % | 39,5 % | **0,57** | 1,11 | 0,78 |
| 10 a 1 | 0,09 | 93,2 % | 27,5 % | 75,0 % | **0,69** | 2,21 | 0,78 |

  Las cifras conservadas muestran menor penalización relativa del modelo en los tres escenarios y mayor carga de revisión al penalizar más los falsos negativos; no constituyen ahorro económico demostrado. El ROC AUC **0,7784** procede de la evaluación previa sobre la prueba original. El AUC-PR **0,5545** (referencia al azar 0,2212) también se conserva como cifra exploratoria pendiente de trazabilidad. No se recalcularon estas métricas en esta revisión.

**Estado de la evaluación:** el conjunto de prueba original ya fue explorado en el EDA del dataset completo y evaluado en el notebook de selección. Sus métricas no constituyen una evaluación final completamente independiente. Las nuevas decisiones de variables, modelo, hiperparámetros, calibración y umbral se tomarán exclusivamente mediante validación dentro del entrenamiento original. Se propone validación cruzada anidada para evaluar el procedimiento de selección; no elimina la exposición previa a los datos. La prueba original solo podrá reportarse como evaluación complementaria con esa limitación explícita. Una evaluación final independiente requeriría datos nuevos no utilizados para decidir.

## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?):** Las columnas `PAY_*`, `BILL_AMT*` y `PAY_AMT*` corresponden a abril–septiembre, antes del mes que se predice, así que no se ve una fuga temporal evidente en ellas. El riesgo también está en el proceso: el EDA ya consultó el dataset completo y la prueba original ya fue evaluada. En adelante, todo ajuste y selección se hará exclusivamente dentro del entrenamiento, sin presentar la prueba consultada como independiente.
- **Sesgos o equidad (¿a quién podría perjudicar el modelo?):** `SEX`, `AGE` y `MARRIAGE` son atributos sensibles; el modelo podría perjudicar a un grupo. Se compararán los errores por grupo en la etapa de validación.
- **Datos personales o confidenciales:** No hay nombres ni documentos, solo un `ID` anónimo, que se excluye del modelo. La licencia del dataset es CC BY 4.0 (verificada en UCI), que permite redistribuirlo citando a Yeh (2009).
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** El modelo ordena por riesgo y prioriza revisiones; no aprueba ni niega crédito. La decisión final sobre un cliente la toma una persona del área de riesgo. Además, los datos son de un solo banco de Taiwán en 2005 y pueden no servir en otro país o época.


## 5b. ¿Qué tipo de modelo tiene sentido para este caso?
Lo que pide el negocio decide el tipo de modelo:

| El negocio necesita… | Entonces el modelo debe… |
|---|---|
| **Detectar al cliente que va a incumplir** (el error caro) | Ser lo más preciso posible al ordenar por riesgo (ROC AUC alto) |
| **Decidir un umbral según costos** | Dar **probabilidades confiables** (bien calibradas), no solo «sí/no» |
| **Justificar por qué se marcó a un cliente** | Poder explicarse (SHAP, o un modelo simple como contraste) |

**Recomendación (con la evidencia actual):**
- **Candidato principal: gradient boosting** (Gradient Boosting, LightGBM o similar). En la exploración previa, la validación cruzada obtiene ≈ 0,78 de ROC AUC frente a 0,73 de la regresión logística. El adelanto del notebook 02 muestra indicios de calibración para HistGradientBoosting, no una garantía para toda la familia ni para el Gradient Boosting optimizado; deberá comprobarse en el modelo concreto antes de aplicar el umbral de costos.
- **Modelo de contraste: regresión logística.** Es menos precisa pero fácil de explicar. Sirve para comprobar que la complejidad del boosting vale la pena.
- **Explicación:** SHAP sobre el modelo principal para decir qué variables llevaron a marcar a un cliente.

**Limitación:** hoy solo se optimizó el Gradient Boosting; los demás corrieron con valores por defecto. La decisión final se toma en el torneo (`05_torneo.ipynb`) con la misma métrica, los mismos folds y ajuste de cada candidato.

## 6. Cierre

- [x] Pregunta, quién decide y costos escritos **con números** (la proporción 5 a 1 es un supuesto por validar con el instructor).
- [x] Línea base calculada (y una regla simple más exigente en el notebook 02).
- [x] Métrica principal elegida y justificada, con métricas de control.
- [x] Tipo de modelo recomendado y su justificación.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué un error cuesta más que el otro.